In [ ]:
# baseline model
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
df=pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])

In [ ]:
df.head()

In [ ]:
# define the preprocessing
nltk.download('wordnet')
nltk.download('stopwords')
def lematization(text):
    lemmatizer=WordNetLemmatizer()
    text=text.split()
    text=[lemmatizer.lemmatize(y) for y in text]
    return " ".join(text)

def remove_stop_words(text):
    stop_words=set(stopwords.words('english'))
    Text=[i for i in str(text).split() if i not in stop_words]
    return " ".join(Text)

def removing_numbers(text):
    text=''.join([i for i in text if not i.isdigit()])
    return text

def lower_case(text):
    text=text.split()
    text=[y.lower() for y in text]
    return " ".join(text)
    
def removing_punctuations(text):
    punctuations =  r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""
    # raw string avoids invalid escape warnings
    text= re.sub('[%s]' % re.escape(punctuations), '', text)

    # remove extra whitespace
    text=re.sub(r'\s+', ' ', text)
    text=' '.join(text.split())
    return text.strip()

def removing_urls(text):
    url_pattern=re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_small_sentence(df):
    for i in range(len(df)):
        if len(df.text.iloc[i].split())<3:
            df.text.iloc[i]=np.nan

def normalize_text(df):
    df.content=df.content.apply(lower_case)
    df.content=df.content.apply(remove_stop_words)
    df.content=df.content.apply( removing_numbers)
    df.content=df.content.apply( removing_punctuations)
    df.content=df.content.apply(removing_urls)
    df.content=df.content.apply(lematization)
    return df


In [ ]:
df=normalize_text(df)
df.head()

In [ ]:
df.sentiment.value_counts()

In [ ]:
x=df.sentiment.isin(['happiness', 'sadness'])
df=df[x]

In [ ]:
df['sentiment']=df['sentiment'].replace({'happiness':1, 'sadness':0})
df.head()

In [ ]:
# apply the Countvectorizer
vectorizer=CountVectorizer(max_features=1000)
X=vectorizer.fit_transform(df['content'])
y=df['sentiment']

In [ ]:
# split data into train, test
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
#
import dagshub

dagshub.init(repo_owner='guptatannu538', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/guptatannu538/mlops-mini-project.mlflow')
mlflow.set_experiment('Logistic Regression Baseline')


In [ ]:
import mlflow
with mlflow.start_run():
    # Log preprocessing parameters
    mlflow.log_param('vectorize', 'Bag of Words')
    mlflow.log_param('num_features', 1000)
    mlflow.log_param('test_size', 0.2)

    # Model building and training
    model=LogisticRegression()
    model.fit(X_train, y_train)
    
    # Log model parameters
    mlflow.log_param('model', 'Logistic Regression')

    # Model evaluation
    y_pred=model.predict(X_test)
    accuracy=accuracy_score(y_test, y_pred)
    precision=precision_score(y_test, y_pred)
    recall=recall_score(y_test, y_pred)
    f1=f1_score(y_test, y_pred)

    # Log evaluation metrics
    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1', f1)

    # Log model
    mlflow.sklearn.log_model(model, 'model')

    # Save and log the notebook
    import os
    notebook_path='exp1_baseline_model.ipynb'
    os.system(f'jupyter nbconvert --to notebook --execute --inplace {notebook_path}')
    mlflow.log_artifact(notebook_path)

    # print the results for verification
    print(f'Accuracy', accuracy)
    print(f'Precision', precision)
    print(f'recall', recall)
    print(f'f1', f1)

